# GeoVision — Exploration Notebook

This notebook walks through the full GeoVision pipeline step by step.

You can run it on **Google Colab** (free) or locally with Jupyter.

**Pipeline:**
1. Install dependencies
2. Download a small set of geotagged images from Wikimedia Commons
3. Extract CNN embeddings with a pretrained ResNet-18
4. Test similarity search on a sample image
5. Run evaluation


## 1 — Install dependencies
(Skip if running locally with venv already activated.)

In [ ]:
# Install CPU-only PyTorch first (faster, smaller)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu -q
!pip install requests Pillow tqdm numpy pandas python-dotenv -q
print('Done.')

## 2 — Clone / mount the project
If on Colab, either clone from GitHub or mount Google Drive.

In [ ]:
import os, sys

# Option A: Clone from GitHub (replace with your repo URL)
# !git clone https://github.com/YOUR_USERNAME/GeoVision.git
# %cd GeoVision

# Option B: If running locally, set the project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
print('Project root:', PROJECT_ROOT)

## 3 — Download geotagged images from Wikimedia Commons

In [ ]:
# Quick demo: download just 5 images from Paris to test the API
import requests, json, io, time
from PIL import Image
from pathlib import Path

WIKIMEDIA_API = 'https://commons.wikimedia.org/w/api.php'
HEADERS = {'User-Agent': 'GeoVisionNotebook/0.1 (educational project)'}

def geosearch(lat, lon, radius=5000, limit=10):
    params = {
        'action': 'query', 'list': 'geosearch',
        'gscoord': f'{lat}|{lon}', 'gsradius': radius,
        'gslimit': limit, 'gsnamespace': 6, 'format': 'json',
    }
    r = requests.get(WIKIMEDIA_API, params=params, headers=HEADERS, timeout=30)
    return r.json().get('query', {}).get('geosearch', [])

def get_thumb_url(title, width=224):
    params = {
        'action': 'query', 'titles': title,
        'prop': 'imageinfo', 'iiprop': 'url|mime',
        'iiurlwidth': width, 'format': 'json',
    }
    r = requests.get(WIKIMEDIA_API, params=params, headers=HEADERS, timeout=30)
    pages = r.json().get('query', {}).get('pages', {})
    for p in pages.values():
        ii = (p.get('imageinfo') or [{}])[0]
        url = ii.get('thumburl') or ii.get('url', '')
        if url and ii.get('mime', '').startswith('image/'):
            return url
    return None

# Try Paris (Eiffel Tower area)
results = geosearch(48.8584, 2.2945, radius=2000, limit=10)
print(f'Found {len(results)} candidate images near Paris Eiffel Tower area')

demo_images = []
for item in results[:5]:
    url = get_thumb_url(item['title'])
    if url:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        img = Image.open(io.BytesIO(resp.content)).convert('RGB')
        demo_images.append({'img': img, 'lat': item['lat'], 'lon': item['lon'], 'title': item['title']})
        time.sleep(0.3)
        print(f'  ✓ {item["title"][:60]}')

print(f'\nDownloaded {len(demo_images)} demo images')

## 4 — Visualise downloaded images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(demo_images), figsize=(4 * len(demo_images), 4))
if len(demo_images) == 1:
    axes = [axes]
for ax, d in zip(axes, demo_images):
    ax.imshow(d['img'])
    ax.set_title(f"lat={d['lat']:.3f}\nlon={d['lon']:.3f}", fontsize=8)
    ax.axis('off')
plt.suptitle('Sample images from Wikimedia Commons (Paris area)')
plt.tight_layout()
plt.show()

## 5 — Extract embeddings with pretrained ResNet-18

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as T
import numpy as np

# Load pretrained ResNet-18 as a feature extractor (no classification head)
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model = torch.nn.Sequential(*list(model.children())[:-1])
model.eval()
print('Model loaded (ResNet-18, CPU, no GPU needed)')

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def embed(img):
    with torch.no_grad():
        vec = model(transform(img).unsqueeze(0)).squeeze().numpy()
    return vec / np.linalg.norm(vec)

# Embed all demo images
for d in demo_images:
    d['embedding'] = embed(d['img'])

print(f'Embedding shape: {demo_images[0]["embedding"].shape}')
print(f'Embedding norm:  {np.linalg.norm(demo_images[0]["embedding"]):.4f}  (should be 1.0)')

## 6 — Similarity search demo

In [ ]:
# Use the first image as a query and find the most similar among the rest
query = demo_images[0]
db    = demo_images[1:]

db_matrix = np.stack([d['embedding'] for d in db])
sims = db_matrix @ query['embedding']
best_idx = int(np.argmax(sims))
best = db[best_idx]

print(f'Query: {query["title"][:60]}')
print(f'Match: {best["title"][:60]}')
print(f'Similarity: {sims[best_idx]:.4f}')
print(f'Predicted coords: {best["lat"]:.5f}, {best["lon"]:.5f}')

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(query['img']); axes[0].set_title('Query'); axes[0].axis('off')
axes[1].imshow(best['img']);
axes[1].set_title(f'Best match (sim={sims[best_idx]:.3f})')
axes[1].axis('off')
plt.show()

## 7 — Run the full pipeline

Once you've confirmed the demo above works, run the full scripts:

```bash
python src/download_data.py   # ~240 images from 8 regions
python src/build_index.py     # extract embeddings and save index
python src/evaluate.py        # held-out evaluation
streamlit run src/app.py      # launch the web UI
```
